## Retrieve Secrets from Azure Key Vault

### Installing Libraries and Utilities

In [ ]:
%pip install azure-keyvault-secrets==4.11.0 azure-identity==1.25.3 openai==2.38.0

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# loading the azure key vault configurations
key_vault_url = os.getenv("KEY_VAULT_URL")
secret_name = os.getenv("KEY_VAULT_SECRET_NAME")

# loading the azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
chat_completions_model = os.getenv("CHAT_COMPLETIONS_MODEL")

### Creating the Key Vault Secret Client

Since this is local development we are using `AzureCliCredential` to authenticate to Azure Key Vault but had it been an Azure Web App or an Azure cloud native workload, you would create a user-assigned managed identity and use its client id in the following manner:
```python
from azure.identity import ManagedIdentityCredential

credential = ManagedIdentityCredential(
    client_id = "YOUR-IDENTITY-CLIENT-ID"
)
```

In [ ]:
from azure.identity import ManagedIdentityCredential
from azure.keyvault.secrets import SecretClient
from azure.identity import AzureCliCredential

# creating the managed identity credential
credential = AzureCliCredential()

# creating the Azure key Vault Secrets client
akv_client = SecretClient(
    vault_url=key_vault_url,
    credential=credential
)

### Fetching the Key Vault Secret

In [ ]:
azure_openai_api_key = akv_client.get_secret(secret_name).value
print(f"fetched openai api key: {azure_openai_api_key}")

### Throwing an API call to the LLM

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
        azure_endpoint = azure_openai_endpoint,
        api_version = "2024-06-01",
        api_key = azure_openai_api_key
)

response = azure_openai_client.chat.completions.create(
        model = chat_completions_model,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful AI assistant"
            },
            {
                "role": "user",
                "content": "What is Azure Key Vault"
            }
        ],
        temperature = 0.7
    )

print(response.choices[0].message.content)